# Phase 04E Metadata Repair

Notebook này sửa một lỗi **metadata-only** trong hai confirmation runs:

- seed 123 và seed 2026 đã thực sự chạy với đúng `SEED`, Teacher và baseline;
- nhưng checkpoint payload và formal complete marker bị hard-code `"seed": 42`;
- tên `PHASE4D_RUN_STATE.json` cũng được chuẩn hóa thành
  `PHASE4E_RUN_STATE.json`.

Notebook không huấn luyện lại, không thay đổi tensor/model/optimizer/RNG và
không mở Held-out Test. Mỗi checkpoint được load tuần tự trên CPU, sửa đúng
một trường metadata, lưu atomic, rồi so sánh fingerprint toàn payload ngoại
trừ trường `seed`.

## Kaggle inputs

1. Output Phase 04E seed 123.
2. Output Phase 04E seed 2026.

Runtime CPU là đủ; Internet Off.


## Cell 1 — Discover và kiểm tra source Phase 04E

Kiểm tra đúng hai run, full source manifest, SHA-256, audit authorization,
comparison cùng seed và trạng thái Test SEALED.


In [1]:
import csv
import hashlib
import json
import os
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path

import torch


CELL_STARTED_AT = time.perf_counter()
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path(
    "/kaggle/working//"
    "phase4_teacher_a_s1_logit_kd_confirmation_metadata_repair"
)
AUDIT_DIR = OUTPUT_ROOT / "audits"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
EXPECTED_SEEDS = (123, 2026)


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target.is_relative_to(root), (root, target)
    return target


SOURCE_RUNS = {}
SOURCE_EVIDENCE = {}
for seed in EXPECTED_SEEDS:
    marker_name = (
        "PHASE4_TEACHER_A_S1_LOGIT_KD_"
        f"SEED{seed}_FULL_RUN_COMPLETE.json"
    )
    marker_matches = sorted(INPUT_ROOT.rglob(marker_name))
    assert len(marker_matches) == 1, (
        marker_name, marker_matches
    )
    marker_path = marker_matches[0].resolve()
    source_root = marker_path.parent
    assert source_root.name == (
        f"phase4_teacher_a_s1_logit_kd_seed{seed}"
    )
    marker = read_json(marker_path)
    assert marker["status"] == "complete"
    assert marker["experiment_id"] == "P4A_A_S1_LOGIT"
    assert int(marker["seed"]) == 42, (
        "Notebook repair chỉ dành cho known packaging bug 42 -> "
        f"{seed}; actual={marker['seed']}"
    )
    assert marker["manifest_verification"] == "all_passed"
    assert marker[
        "ready_for_phase04f_three_seed_aggregate"
    ] is True
    assert marker["test_set_status"] == "sealed"
    assert marker["test_set_used"] is False

    authorization = read_json(
        source_root / "audits/phase4e_cell01_authorization.json"
    )
    assert authorization["status"] == "passed"
    assert int(authorization["confirmation_seed"]) == seed
    assert authorization["method_changes_allowed"] is False
    assert authorization["test_set_status"] == "sealed"
    assert authorization["test_set_used"] is False

    comparison_path = source_root / (
        "results/teacher_a_s1_logit_kd_"
        f"seed{seed}_vs_baseline.json"
    )
    comparison = read_json(comparison_path)
    assert f"baseline_seed{seed}" in comparison
    assert f"kd_seed{seed}" in comparison
    assert comparison["kd_improves_cider_by_0_001"] is True
    assert comparison["selection_split"] == "validation"
    assert comparison["test_set_status"] == "sealed"
    assert comparison["test_set_used"] is False

    manifest_path = resolve_inside(
        source_root,
        marker["manifest"]["csv"]["relative_path"],
    )
    assert sha256_file(manifest_path) == (
        marker["manifest"]["csv"]["sha256"]
    )
    manifest_rows = list(csv.DictReader(
        manifest_path.open("r", encoding="utf-8", newline="")
    ))
    assert len(manifest_rows) == int(marker["manifest_rows"])
    sum_lines = []
    for row in manifest_rows:
        artifact = resolve_inside(
            source_root, row["relative_path"]
        )
        assert artifact.stat().st_size == int(row["size_bytes"])
        assert sha256_file(artifact) == row["sha256"]
        sum_lines.append(
            f'{row["sha256"]}  {row["relative_path"]}'
        )
    sums_path = resolve_inside(
        source_root,
        marker["manifest"]["sha256sums"]["relative_path"],
    )
    assert sha256_file(sums_path) == (
        marker["manifest"]["sha256sums"]["sha256"]
    )
    actual_sum_lines = [
        line.strip()
        for line in sums_path.read_text(
            encoding="utf-8"
        ).splitlines()
        if line.strip()
    ]
    assert actual_sum_lines == sum_lines

    SOURCE_RUNS[seed] = {
        "root": source_root,
        "marker_path": marker_path,
        "marker": marker,
        "comparison": comparison,
        "manifest_rows": manifest_rows,
    }
    SOURCE_EVIDENCE[seed] = {
        "source_root": str(source_root),
        "source_marker_sha256": sha256_file(marker_path),
        "source_manifest_sha256": sha256_file(manifest_path),
        "source_manifest_rows": len(manifest_rows),
        "authorization_confirmation_seed": int(
            authorization["confirmation_seed"]
        ),
        "original_incorrect_marker_seed": int(marker["seed"]),
        "target_seed": seed,
    }

write_json(
    AUDIT_DIR / "phase4e_repair_cell01_source_audit.json",
    {
        "audit_version": "phase4e_repair_cell01_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "source_runs": SOURCE_EVIDENCE,
        "source_manifest_verification": "all_passed",
        "known_issue": (
            "checkpoint and marker seed metadata hard-coded to 42"
        ),
        "training_validity": (
            "confirmed by seed-specific authorization audit, "
            "Teacher/baseline selection and result paths"
        ),
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04E REPAIR CELL 1 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Source runs               : 2 / 2 VERIFIED")
print("Source manifest rows      :", {
    seed: len(SOURCE_RUNS[seed]["manifest_rows"])
    for seed in EXPECTED_SEEDS
})
print("Actual run seeds          : 123 / 2026 VERIFIED BY AUDIT")
print("Incorrect packaged seed   : 42 / 42 CONFIRMED")
print("Test Set status           : SEALED")
print("Next                      : copy and repair checkpoints")


PHASE 04E REPAIR CELL 1 STATUS
Status                    : PASSED
Source runs               : 2 / 2 VERIFIED
Source manifest rows      : {123: 52, 2026: 52}
Actual run seeds          : 123 / 2026 VERIFIED BY AUDIT
Incorrect packaged seed   : 42 / 42 CONFIRMED
Test Set status           : SEALED
Next                      : copy and repair checkpoints


## Cell 2 — Copy artifacts và sửa checkpoint metadata

Tạo canonical output mới. Các checkpoint được sửa atomic; fingerprint của
mọi trường ngoại trừ `seed` phải giống hệt trước và sau.


In [2]:
CELL_STARTED_AT = time.perf_counter()


def update_digest(digest, value):
    if isinstance(value, torch.Tensor):
        tensor = value.detach().cpu().contiguous()
        digest.update(b"TENSOR")
        digest.update(str(tensor.dtype).encode())
        digest.update(str(tuple(tensor.shape)).encode())
        digest.update(
            tensor.reshape(-1).view(torch.uint8).numpy().tobytes()
        )
    elif isinstance(value, dict):
        digest.update(b"DICT")
        for key in sorted(value, key=lambda item: repr(item)):
            update_digest(digest, key)
            update_digest(digest, value[key])
    elif isinstance(value, (list, tuple)):
        digest.update(
            b"LIST" if isinstance(value, list) else b"TUPLE"
        )
        for item in value:
            update_digest(digest, item)
    elif value is None:
        digest.update(b"NONE")
    elif isinstance(value, bytes):
        digest.update(b"BYTES")
        digest.update(value)
    else:
        digest.update(type(value).__name__.encode())
        digest.update(repr(value).encode("utf-8"))


def payload_digest_excluding_seed(payload):
    digest = hashlib.sha256()
    update_digest(
        digest,
        {
            key: value
            for key, value in payload.items()
            if key != "seed"
        },
    )
    return digest.hexdigest()


REPAIRED_RUNS = {}
for seed in EXPECTED_SEEDS:
    source = SOURCE_RUNS[seed]
    source_root = source["root"]
    target_root = OUTPUT_ROOT / source_root.name
    target_root.mkdir(parents=True, exist_ok=True)

    skip_names = {
        f"PHASE4_TEACHER_A_S1_LOGIT_KD_SEED{seed}_"
        "FULL_RUN_COMPLETE.json",
        "PHASE4D_RUN_STATE.json",
    }
    for source_path in sorted(source_root.rglob("*")):
        if not source_path.is_file():
            continue
        relative = source_path.relative_to(source_root)
        if relative.parts[0] == "manifests":
            continue
        if source_path.name in skip_names:
            continue
        if source_path.suffix == ".pt":
            continue
        target_path = target_root / relative
        target_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source_path, target_path)

    source_run_state = read_json(
        source_root / "PHASE4D_RUN_STATE.json"
    )
    assert source_run_state["schema_version"] == (
        "phase4e_run_state_v1"
    )
    assert source_run_state["status"] == "complete"
    source_run_state["confirmation_seed"] = seed
    source_run_state["metadata_repaired"] = True
    source_run_state["metadata_repair_note"] = (
        "Renamed from PHASE4D_RUN_STATE.json and annotated; "
        "training state unchanged."
    )
    write_json(
        target_root / "PHASE4E_RUN_STATE.json",
        source_run_state,
    )

    checkpoint_audits = {}
    for role in ("best", "last"):
        checkpoint_name = (
            "teacher_a_s1_logit_kd_"
            f"seed{seed}_{role}.pt"
        )
        source_checkpoint = (
            source_root / "checkpoints" / checkpoint_name
        )
        target_checkpoint = (
            target_root / "checkpoints" / checkpoint_name
        )
        target_checkpoint.parent.mkdir(
            parents=True, exist_ok=True
        )
        payload = torch.load(
            source_checkpoint,
            map_location="cpu",
            weights_only=False,
        )
        assert payload["schema_version"] == (
            "phase4_teacher_a_s1_kd_checkpoint_v1"
        )
        assert payload["experiment_id"] == "P4A_A_S1_LOGIT"
        assert int(payload["seed"]) == 42
        assert payload["test_set_status"] == "sealed"
        assert payload["test_set_used"] is False
        before_digest = payload_digest_excluding_seed(payload)
        payload["seed"] = seed
        temporary = target_checkpoint.with_suffix(".pt.tmp")
        torch.save(payload, temporary)
        os.replace(temporary, target_checkpoint)
        del payload

        verified = torch.load(
            target_checkpoint,
            map_location="cpu",
            weights_only=False,
        )
        assert int(verified["seed"]) == seed
        assert payload_digest_excluding_seed(
            verified
        ) == before_digest
        del verified
        checkpoint_audits[role] = {
            "source_relative_path": (
                source_checkpoint.relative_to(
                    source_root
                ).as_posix()
            ),
            "source_sha256": sha256_file(source_checkpoint),
            "target_relative_path": (
                target_checkpoint.relative_to(
                    target_root
                ).as_posix()
            ),
            "target_sha256": sha256_file(target_checkpoint),
            "original_seed": 42,
            "corrected_seed": seed,
            "payload_digest_excluding_seed": before_digest,
            "non_seed_payload_exact": True,
        }

    repair_audit_path = target_root / (
        "audits/phase4e_confirmation_seed_metadata_repair.json"
    )
    repair_audit = {
        "audit_version": (
            "phase4e_confirmation_seed_metadata_repair_v1"
        ),
        "created_at_utc": now_utc(),
        "status": "passed",
        "experiment_id": "P4A_A_S1_LOGIT",
        "target_seed": seed,
        "original_incorrect_seed": 42,
        "repair_scope": [
            "best checkpoint seed metadata",
            "last checkpoint seed metadata",
            "formal completion marker seed metadata",
            "run-state filename and confirmation_seed annotation",
            "manifest regeneration",
        ],
        "training_rerun": False,
        "model_tensor_changed": False,
        "optimizer_state_changed": False,
        "scheduler_state_changed": False,
        "gradscaler_state_changed": False,
        "rng_state_changed": False,
        "checkpoints": checkpoint_audits,
        "source_marker_sha256": SOURCE_EVIDENCE[seed][
            "source_marker_sha256"
        ],
        "test_set_status": "sealed",
        "test_set_used": False,
    }
    write_json(repair_audit_path, repair_audit)

    REPAIRED_RUNS[seed] = {
        "root": target_root,
        "checkpoint_audits": checkpoint_audits,
        "repair_audit_path": repair_audit_path,
    }

print("=" * 72)
print("PHASE 04E REPAIR CELL 2 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Checkpoints repaired      : 4 / 4")
print("Non-seed payload exact    : 4 / 4 PASSED")
print("Training rerun            : False")
print("Model/optimizer/RNG change: False")
print("Test Set status           : SEALED")
print("Next                      : rebuild formal manifests")


PHASE 04E REPAIR CELL 2 STATUS
Status                    : PASSED
Checkpoints repaired      : 4 / 4
Non-seed payload exact    : 4 / 4 PASSED
Training rerun            : False
Model/optimizer/RNG change: False
Test Set status           : SEALED
Next                      : rebuild formal manifests


## Cell 3 — Rebuild manifests và corrected complete markers

Phát hành hai run canonical đã sửa và một authorization marker cho Phase 04F.


In [3]:
CELL_STARTED_AT = time.perf_counter()
CORRECTED_MARKERS = {}

for seed in EXPECTED_SEEDS:
    target_root = REPAIRED_RUNS[seed]["root"]
    run_manifest_dir = target_root / "manifests"
    run_manifest_dir.mkdir(parents=True, exist_ok=True)
    manifest_csv = run_manifest_dir / "phase4e_repaired_manifest.csv"
    manifest_json = run_manifest_dir / "phase4e_repaired_manifest.json"
    sha256sums = run_manifest_dir / (
        "phase4e_repaired_SHA256SUMS.txt"
    )
    marker_path = target_root / (
        "PHASE4_TEACHER_A_S1_LOGIT_KD_"
        f"SEED{seed}_FULL_RUN_COMPLETE.json"
    )

    artifact_paths = sorted(
        [
            path for path in target_root.rglob("*")
            if path.is_file()
            and path not in {
                manifest_csv, manifest_json, sha256sums, marker_path
            }
            and not path.name.endswith(".tmp")
        ],
        key=lambda path: path.relative_to(
            target_root
        ).as_posix(),
    )
    rows = [
        {
            "relative_path": path.relative_to(
                target_root
            ).as_posix(),
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }
        for path in artifact_paths
    ]
    with manifest_csv.open(
        "w", encoding="utf-8", newline=""
    ) as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=[
                "relative_path", "size_bytes", "sha256"
            ],
        )
        writer.writeheader()
        writer.writerows(rows)
    write_json(
        manifest_json,
        {
            "schema_version": (
                "phase4e_repaired_manifest_v1"
            ),
            "created_at_utc": now_utc(),
            "artifact_count": len(rows),
            "artifacts": rows,
        },
    )
    sha256sums.write_text(
        "".join(
            f'{row["sha256"]}  {row["relative_path"]}\n'
            for row in rows
        ),
        encoding="utf-8",
    )

    source_marker = SOURCE_RUNS[seed]["marker"]
    corrected_marker = dict(source_marker)
    corrected_marker["marker_version"] = (
        "phase4_teacher_a_s1_logit_kd_"
        f"seed{seed}_full_run_complete_metadata_repaired_v1"
    )
    corrected_marker["created_at_utc"] = now_utc()
    corrected_marker["seed"] = seed
    corrected_marker["last_checkpoint"] = {
        "relative_path": (
            "checkpoints/teacher_a_s1_logit_kd_"
            f"seed{seed}_last.pt"
        ),
        "sha256": REPAIRED_RUNS[seed][
            "checkpoint_audits"
        ]["last"]["target_sha256"],
    }
    corrected_marker["best_checkpoint"] = {
        "relative_path": (
            "checkpoints/teacher_a_s1_logit_kd_"
            f"seed{seed}_best.pt"
        ),
        "sha256": REPAIRED_RUNS[seed][
            "checkpoint_audits"
        ]["best"]["target_sha256"],
    }
    corrected_marker["metadata_repair"] = {
        "status": "passed",
        "training_rerun": False,
        "only_seed_metadata_changed_in_checkpoints": True,
        "non_seed_checkpoint_payload_exact": True,
        "audit_relative_path": (
            REPAIRED_RUNS[seed]["repair_audit_path"]
            .relative_to(target_root)
            .as_posix()
        ),
        "audit_sha256": sha256_file(
            REPAIRED_RUNS[seed]["repair_audit_path"]
        ),
        "source_marker_sha256": SOURCE_EVIDENCE[seed][
            "source_marker_sha256"
        ],
    }
    corrected_marker["manifest"] = {
        "csv": {
            "relative_path": manifest_csv.relative_to(
                target_root
            ).as_posix(),
            "sha256": sha256_file(manifest_csv),
        },
        "json": {
            "relative_path": manifest_json.relative_to(
                target_root
            ).as_posix(),
            "sha256": sha256_file(manifest_json),
        },
        "sha256sums": {
            "relative_path": sha256sums.relative_to(
                target_root
            ).as_posix(),
            "sha256": sha256_file(sha256sums),
        },
    }
    corrected_marker["manifest_rows"] = len(rows)
    corrected_marker["manifest_verification"] = "all_passed"
    corrected_marker[
        "ready_for_phase04f_three_seed_aggregate"
    ] = True
    corrected_marker["test_set_status"] = "sealed"
    corrected_marker["test_set_used"] = False
    write_json(marker_path, corrected_marker)
    CORRECTED_MARKERS[seed] = {
        "path": marker_path,
        "payload": corrected_marker,
        "sha256": sha256_file(marker_path),
    }

AUTHORIZATION_MARKER = OUTPUT_ROOT / (
    "PHASE4_TEACHER_A_S1_LOGIT_KD_"
    "THREE_SEED_AGGREGATE_AUTHORIZED.json"
)
write_json(
    AUTHORIZATION_MARKER,
    {
        "marker_version": (
            "phase4_teacher_a_s1_logit_kd_"
            "three_seed_aggregate_authorized_v1"
        ),
        "created_at_utc": now_utc(),
        "status": "authorized",
        "experiment_id": "P4A_A_S1_LOGIT",
        "development_seed": 42,
        "confirmation_seeds": [123, 2026],
        "corrected_confirmation_markers": {
            str(seed): {
                "relative_path": (
                    CORRECTED_MARKERS[seed]["path"]
                    .relative_to(OUTPUT_ROOT)
                    .as_posix()
                ),
                "sha256": CORRECTED_MARKERS[seed]["sha256"],
            }
            for seed in EXPECTED_SEEDS
        },
        "method_changes_allowed": False,
        "next_phase": (
            "Phase 04F Teacher A -> S1 Logit-KD "
            "three-seed aggregate"
        ),
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

print("=" * 72)
print("PHASE 04E REPAIR CELL 3 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Corrected seed markers    : 123 / 2026")
print("Run manifests rebuilt     : 2 / 2")
print("Phase 04F authorization   : CREATED")
print("Training rerun            : False")
print("Test Set status           : SEALED")
print("Next                      : final integrity verification")


PHASE 04E REPAIR CELL 3 STATUS
Status                    : PASSED
Corrected seed markers    : 123 / 2026
Run manifests rebuilt     : 2 / 2
Phase 04F authorization   : CREATED
Training rerun            : False
Test Set status           : SEALED
Next                      : final integrity verification


## Cell 4 — Final integrity gate

Đọc lại toàn bộ canonical manifests và checkpoint headers, sau đó phát hành
formal repair-complete marker.


In [4]:
CELL_STARTED_AT = time.perf_counter()
FINAL_SUMMARY = {}

for seed in EXPECTED_SEEDS:
    marker_path = CORRECTED_MARKERS[seed]["path"]
    marker = read_json(marker_path)
    target_root = marker_path.parent
    assert marker["status"] == "complete"
    assert int(marker["seed"]) == seed
    assert marker["experiment_id"] == "P4A_A_S1_LOGIT"
    assert marker["metadata_repair"]["status"] == "passed"
    assert marker["test_set_status"] == "sealed"
    assert marker["test_set_used"] is False

    manifest_path = resolve_inside(
        target_root,
        marker["manifest"]["csv"]["relative_path"],
    )
    assert sha256_file(manifest_path) == (
        marker["manifest"]["csv"]["sha256"]
    )
    rows = list(csv.DictReader(
        manifest_path.open("r", encoding="utf-8", newline="")
    ))
    assert len(rows) == int(marker["manifest_rows"])
    for row in rows:
        artifact = resolve_inside(
            target_root, row["relative_path"]
        )
        assert artifact.stat().st_size == int(row["size_bytes"])
        assert sha256_file(artifact) == row["sha256"]

    for role in ("best", "last"):
        checkpoint = resolve_inside(
            target_root,
            marker[f"{role}_checkpoint"]["relative_path"],
        )
        assert sha256_file(checkpoint) == (
            marker[f"{role}_checkpoint"]["sha256"]
        )
        payload = torch.load(
            checkpoint,
            map_location="cpu",
            weights_only=False,
        )
        assert int(payload["seed"]) == seed
        assert payload["experiment_id"] == "P4A_A_S1_LOGIT"
        assert payload["test_set_status"] == "sealed"
        assert payload["test_set_used"] is False
        del payload

    comparison = marker["comparison"]
    FINAL_SUMMARY[seed] = {
        "best_epoch": int(
            comparison[f"kd_seed{seed}"]["best_epoch"]
        ),
        "baseline_cider_d": float(
            comparison[f"baseline_seed{seed}"]["CIDEr-D"]
        ),
        "kd_cider_d": float(
            comparison[f"kd_seed{seed}"]["CIDEr-D"]
        ),
        "delta_cider_d": float(
            comparison["delta"]["CIDEr-D"]
        ),
        "marker_sha256": sha256_file(marker_path),
        "manifest_rows": len(rows),
    }

COMPLETE_MARKER = OUTPUT_ROOT / (
    "PHASE4E_CONFIRMATION_SEED_METADATA_REPAIR_COMPLETE.json"
)
write_json(
    COMPLETE_MARKER,
    {
        "marker_version": (
            "phase4e_confirmation_seed_metadata_"
            "repair_complete_v1"
        ),
        "created_at_utc": now_utc(),
        "status": "complete",
        "experiment_id": "P4A_A_S1_LOGIT",
        "repaired_seeds": [123, 2026],
        "summary": FINAL_SUMMARY,
        "checkpoint_headers_verified": "4 / 4",
        "non_seed_checkpoint_payload_exact": "4 / 4",
        "run_manifests_verified": "2 / 2",
        "training_rerun": False,
        "phase04f_authorized": True,
        "authorization_marker": {
            "relative_path": AUTHORIZATION_MARKER.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(AUTHORIZATION_MARKER),
        },
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

print("=" * 72)
print("PHASE 04E METADATA REPAIR COMPLETE")
print("=" * 72)
print("Status                    : COMPLETE")
for seed in EXPECTED_SEEDS:
    row = FINAL_SUMMARY[seed]
    print(
        f"Seed {seed:<4} CIDEr-D       : "
        f"{row['baseline_cider_d']:.6f} -> "
        f"{row['kd_cider_d']:.6f} "
        f"(delta={row['delta_cider_d']:+.6f})"
    )
print("Checkpoint headers        : 4 / 4 VERIFIED")
print("Non-seed payload exact    : 4 / 4 PASSED")
print("Run manifests             : 2 / 2 VERIFIED")
print("Training rerun            : False")
print("Test Set status           : SEALED")
print("Phase 04F                 : AUTHORIZED")
print("Marker                    :", COMPLETE_MARKER)


PHASE 04E METADATA REPAIR COMPLETE
Status                    : COMPLETE
Seed 123  CIDEr-D       : 0.987251 -> 1.006517 (delta=+0.019266)
Seed 2026 CIDEr-D       : 0.978451 -> 1.006557 (delta=+0.028107)
Checkpoint headers        : 4 / 4 VERIFIED
Non-seed payload exact    : 4 / 4 PASSED
Run manifests             : 2 / 2 VERIFIED
Training rerun            : False
Test Set status           : SEALED
Phase 04F                 : AUTHORIZED
Marker                    : /kaggle/working//phase4_teacher_a_s1_logit_kd_confirmation_metadata_repair/PHASE4E_CONFIRMATION_SEED_METADATA_REPAIR_COMPLETE.json
